# Dynamic Phi Model Training with GRPO

This notebook implements the same training process as the dynamic_phi.py script, allowing for interactive execution and visualization of the training process.

## Setup Logging

First, let's set up logging to track our progress.

In [ ]:
import os

# Set GPU device
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print(f"Using GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")


Using GPU: 0


In [ ]:
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
import os
import wandb
import logging
import json
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from datetime import datetime
from unsloth import is_bfloat16_supported
import sys
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback


sys.path.append("/Home/stat/laschos/math/AIMO2_initial")
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from grpo.config import RewardConfig
from grpo.dynamic_reward import DynamicReward
from utils.similarity_checker import SolutionSimilarityChecker
from utils.data_preparation import prepare_combined_data
from utils.agents import (
    FULLSOLUTION_SYSTEM_PROMPT, 
    FINALIZATION_SYSTEM_PROMPT,
    PROGRAMMER_SYSTEM_PROMPT,
    TUTOR_SYSTEM_PROMPT
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
def setup_logging(model_type: str) -> logging.Logger:
    """Setup logging configuration"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"logs/{model_type}"
    os.makedirs(log_dir, exist_ok=True)
    
    logger = logging.getLogger('dynamic_grpo')
    
    # Clear any existing handlers to prevent duplicate logging
    if logger.handlers:
        logger.handlers.clear()
        
    logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(
        f"{log_dir}/training_{timestamp}.log"
    )
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    logger.addHandler(file_handler)
    logger.addHandler(logging.StreamHandler())
    return logger

class LoggingCallback(TrainerCallback):
    """Callback for logging training metrics"""
    def __init__(self, reward_func, logger, save_frequency=100):
        self.reward_func = reward_func
        self.save_frequency = save_frequency
        self.step = 0
        self.logger = logger
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        self.step += 1
        
        if logs and 'rewards/0' in logs and hasattr(self.reward_func, 'stats'):
            # Print detailed stats to console/log file
            self.logger.info("\n" + "="*50)
            self.logger.info(f"Step {self.step} - Reward Stats Summary:")
            
            # Get and log the stats summary
            stats_summary = self.reward_func.stats.get_summary()
            self.logger.info(stats_summary)
            self.logger.info("="*50 + "\n")
            
            # Key performance metrics for wandb
            wandb_stats = {
                'reward': logs['rewards/0'],
                'average_reward': self.reward_func.stats.reward_components.get('average_reward', 0.0),
                'total_batches': self.reward_func.stats.total_batches,
                'total_examples': self.reward_func.stats.total_examples
            }
            
            # Add dynamic reward specific metrics
            if 'solution_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['solution_reward_uses'] = self.reward_func.stats.reward_components['solution_reward_uses']
            if 'completion_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['completion_reward_uses'] = self.reward_func.stats.reward_components['completion_reward_uses']
                
            # Track example types in the batch
            if hasattr(state, 'train_dataloader') and state.train_dataloader is not None:
                try:
                    # Get current batch
                    batch_idx = (state.global_step - 1) % len(state.train_dataloader)
                    current_batch = list(state.train_dataloader)[batch_idx]
                    
                    # Count example types if available
                    if 'example_type' in current_batch:
                        example_types = current_batch['example_type']
                        solution_count = sum(1 for t in example_types if t == 'solution')
                        completion_count = sum(1 for t in example_types if t == 'completion')
                        wait_count = sum(1 for t in example_types if t == 'wait')
                        
                        wandb_stats['solution_examples'] = solution_count
                        wandb_stats['completion_examples'] = completion_count
                        wandb_stats['wait_examples'] = wait_count
                except Exception as e:
                    self.logger.warning(f"Could not track example types: {str(e)}")
            
            # Add all stats from reward_components to wandb
            for key, value in self.reward_func.stats.reward_components.items():
                wandb_stats[f'reward_components/{key}'] = value
                
            # Add group stats
            for key, value in self.reward_func.stats.group_stats.items():
                wandb_stats[f'group_stats/{key}'] = value
                
            # Add step stats
            for key, value in self.reward_func.stats.step_stats.items():
                wandb_stats[f'step_stats/{key}'] = value
                
            # Add similarity stats
            for key, value in self.reward_func.stats.similarity_stats.items():
                wandb_stats[f'similarity_stats/{key}'] = value
                
            # Add programming stats
            for key, value in self.reward_func.stats.programming_stats.items():
                wandb_stats[f'programming_stats/{key}'] = value
                
            # Add reward distribution
            if hasattr(self.reward_func.stats, 'reward_distribution') and self.reward_func.stats.reward_distribution:
                # Only log the top 10 most common rewards to avoid cluttering wandb
                sorted_rewards = sorted(
                    self.reward_func.stats.reward_distribution.items(), 
                    key=lambda x: self.reward_func.stats.reward_distribution[x[0]], 
                    reverse=True
                )[:10]
                
                for reward, count in sorted_rewards:
                    wandb_stats[f'reward_distribution/{reward}'] = count
            
            # Update logs with our metrics
            logs.update(wandb_stats)

## Main Training Setup

Now let's set up the main training configuration and components.

In [4]:
# Configuration
# Configuration
model_type = "dynamic_1"
model_name = "unsloth/Phi-4"
dataset_name = "Metaskepsis/Olympiads_hard_filtered"

# Setup logging first
logger = setup_logging(model_type)

# Initialize config
reward_config = RewardConfig(model_type=model_type)

# Setup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"train_results/{reward_config.model_type}/{timestamp}"
wandbname = f"{model_type}, DB={reward_config.group_diversity_bonus}, {model_name}, {dataset_name}, {timestamp}"

# Initialize wandb
wandb.init(
    project="grpo",
    name=wandbname,
    config={
        "model_type": reward_config.model_type,
        "dataset": dataset_name,
        "base_reward": 3.0,
        "diversity_bonus": 0.3,
        "step_continuity_reward": 0.5
    }
)

# Initialize similarity checker first
similarity_checker = SolutionSimilarityChecker(reward_config)

# Initialize dynamic reward function
reward_func = DynamicReward(reward_config, similarity_checker)
logger.info("\nInitialized DynamicReward:")
logger.info(f"Has stats object: {hasattr(reward_func, 'stats')}")

# Print initial stats configuration
if hasattr(reward_func, 'stats'):
    logger.info("Initial stats configuration:")
    for category in ['reward_components', 'group_stats', 'step_stats', 'similarity_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            logger.info(f"{category}: {stats_dict}")
else:
    logger.warning("No stats object found in reward_func!")

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: artnoage (metaskepsis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Initialized DynamicReward:
Has stats object: True
Initial stats configuration:
reward_components: {'base_rewards': 0, 'step_continuity_rewards': 0, 'diversity_bonuses': 0, 'similarity_penalties': 0, 'validation_rewards': 0, 'total_length_penalty': 0.0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_rewards': 0.0, 'average_reward': 0.0, 'solution_reward_uses': 0, 'finalization_reward_uses': 0, 'programming_reward_uses': 0, 'tutor_reward_uses': 0, 'correct_verdict_rewards': 0, 'correct_fix_rewards': 0, 'syntax_rewards': 0, 'execution_rewards': 0, 'correctness_rewards': 0, 'syntax_valid_solutions': 0, 'execution_valid_solutions': 0}
group_stats: {'unique_solutions': 0, 'similar_solutions': 0, 'correct_answers': 0, 'incorrect_answers': 0, 'total_similarity': 0.0, 'diversity_bonuses': 0, 'similarity_penalties': 0}
step_stats: {'correct_step_numbering': 0, 'incorrect_step_numbering': 0, 'total_steps_completed': 0}
similarity_stats: {'unique_completions': 0, 'similar_completions': 0, '

In [5]:
# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=3300,
    fast_inference=True,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
    gpu_memory_utilization=0.75,
    max_lora_rank=64)
    
# Function to count tokens in a string
def count_tokens(text):
    return len(tokenizer.encode(text))
    
# Calculate token counts for system prompts
solver_prompt_tokens = count_tokens(FULLSOLUTION_SYSTEM_PROMPT)
completion_prompt_tokens = count_tokens(FINALIZATION_SYSTEM_PROMPT)
logger.info(f"Solver system prompt: {solver_prompt_tokens} tokens")
logger.info(f"Completion system prompt: {completion_prompt_tokens} tokens")

# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)
    


INFO 03-09 21:02:44 __init__.py:207] Automatically detected platform cuda.
==((====))==  Unsloth 2025.3.8: Fast Llama patching. Transformers: 4.49.0. vLLM: 0.7.3.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.393 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu124. CUDA: 8.0. CUDA Toolkit: 12.4. Triton: 3.1.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/Phi-4 with actual GPU utilization = 74.13%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.39 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 3300. Num Sequences = 128.
Unsloth: vLLM's KV Cache can use up to 1.41 GB. Also swap space = 6 GB.
INFO 03-09 21:02:56 config.py:549] This model supports multiple tasks: {'generate', 'score', 'embed', 'classify', 'reward'}. Defaulting 

[W309 21:02:58.499774138 CUDAAllocatorConfig.h:28] Warning: expandable_segments not supported on this platform (function operator())


INFO 03-09 21:02:59 weight_utils.py:254] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/6 [00:00<?, ?it/s]


INFO 03-09 21:03:06 model_runner.py:1115] Loading model weights took 27.4110 GB
INFO 03-09 21:03:06 punica_selector.py:18] Using PunicaWrapperGPU.
INFO 03-09 21:03:17 worker.py:267] Memory profiling takes 7.67 seconds
INFO 03-09 21:03:17 worker.py:267] the current vLLM instance can use total_gpu_memory (39.39GiB) x gpu_memory_utilization (0.74) = 29.20GiB
INFO 03-09 21:03:17 worker.py:267] model weights take 27.41GiB; non_torch_memory takes 0.09GiB; PyTorch activation peak memory takes 0.50GiB; the rest of the memory reserved for KV Cache is 1.20GiB.
INFO 03-09 21:03:17 executor_base.py:111] # cuda blocks: 394, # CPU blocks: 1966
INFO 03-09 21:03:17 executor_base.py:116] Maximum concurrency for 3300 tokens per request: 1.91x
INFO 03-09 21:03:34 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error oc

Capturing CUDA graph shapes: 100%|████████████████████████████████████████████| 19/19 [01:06<00:00,  3.50s/it]

INFO 03-09 21:04:41 model_runner.py:1562] Graph capturing finished in 67 secs, took 1.47 GiB
INFO 03-09 21:04:41 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 91.70 seconds



Solver system prompt: 150 tokens
Completion system prompt: 285 tokens
Unsloth 2025.3.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


## Load Dataset

Now let's start the training process.

In [6]:
def get_questions(split="train") -> Dataset:
    """Load and format dataset with full solution, completion, programming, and wait examples
    with the following distribution:
    - 35% solution examples
    - 35% programming examples
    - 15% completion examples
    - 15% wait examples
    """
    
    
    # Load the base dataset
    data = load_dataset(dataset_name,split="train")
    
    # Define the distribution
    # You can set any value to 0 to skip generating that type of example
    distribution = {
        'solution': 0.5,
        'programming': 0.5,
        'finalization': 0.0,
        'tutor': 0.0
    }
    
    # Use the prepare_combined_data function with all system prompts
    return prepare_combined_data(
        data, 
        FULLSOLUTION_SYSTEM_PROMPT, 
        FINALIZATION_SYSTEM_PROMPT, 
        PROGRAMMER_SYSTEM_PROMPT,
        TUTOR_SYSTEM_PROMPT,
        tokenizer, 
        distribution)

# Get the formatted dataset with all types of examples
formatted_dataset = get_questions()
# Shuffle the combined dataset
formatted_dataset = formatted_dataset.shuffle(seed=20)
# Use a reasonable number of examples
formatted_dataset = formatted_dataset.select(range(2000))

Dataset has 0 examples with model_solutions
Creating solution examples...
Creating programming examples...
Created 20672 full solution examples (target: 10336)
Created 20672 programming examples (target: 10336)
Dataset type distribution before combining:
Solution dataset: {'solution': 10336}
Programming dataset: {'programming': 10336}
Combined dataset types: {'solution': 10336, 'programming': 10336}
Type percentages: {'solution': '50.0%', 'programming': '50.0%'}


## Create confing and trainer

Now let's start the training process.

In [7]:
# GRPO specific training arguments
training_args = GRPOConfig(
    torch_empty_cache_steps=1,
    learning_rate=6e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=1,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=7,
    gradient_accumulation_steps=4,
    num_generations=7,
    max_prompt_length=800,
    max_completion_length=2500,
    num_train_epochs=1,
    save_steps=50,
    max_grad_norm=0.1,
    report_to="wandb",
    output_dir=output_dir,
)

# Log the dataset structure before training
logger.info("Dataset structure before training:")
sample_example = formatted_dataset[0]
for key, value in sample_example.items():
    logger.info(f"  {key}: {type(value)} - {value}")

# Initialize trainer with reward function
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_func],
    args=training_args,
    train_dataset=formatted_dataset,
    callbacks=[LoggingCallback(reward_func=reward_func, logger=logger, save_frequency=10)]
)

 # Log dataset information before training
logger.info("Dataset information before training:")
logger.info(f"Total examples: {len(formatted_dataset)}")

# Count example types in the dataset
example_types = {}
for example in formatted_dataset:
    et = example.get('example_type', 'unknown')
    example_types[et] = example_types.get(et, 0) + 1

logger.info(f"Example types in dataset: {example_types}")

# Log a sample batch structure
sample_batch = {
    'prompt': [formatted_dataset[i]['prompt'] for i in range(min(3, len(formatted_dataset)))],
    'answer': [formatted_dataset[i]['answer'] for i in range(min(3, len(formatted_dataset)))],
    'example_type': [formatted_dataset[i]['example_type'] for i in range(min(3, len(formatted_dataset)))]
}

logger.info("Sample batch structure:")
for key, value in sample_batch.items():
    if key != 'prompt':  # Skip logging the full prompts
        logger.info(f"  {key}: {value}")

# The example_type is already in the dataset, no need to add it again
# Just verify that it's present in all examples
example_type_missing = sum(1 for example in formatted_dataset if "example_type" not in example)
if example_type_missing > 0:
    logger.warning(f"Found {example_type_missing} examples without example_type field")
else:
    logger.info("All examples have example_type field correctly set")

# Print a few examples to verify example_type is set correctly
for i in range(min(5, len(formatted_dataset))):
    logger.info(f"Example {i} type: {formatted_dataset[i]['example_type']}")

Dataset structure before training:
  id: <class 'int'> - 4340
  problem: <class 'str'> - Let \( p, q, r \) be the three sides of triangle \( PQR \). If \( p^{4} + q^{4} + r^{4} = 2r^{2}(p^{2} + q^{2}) \), find \( a \), where \( a = \cos^{2} R \) and \( R \) denotes the angle opposite \( r \).
  solution: <class 'str'> - 

Given:

\[
p^4 + q^4 + r^4 = 2r^2(p^2 + q^2)
\]

We need to find \( a \), where \( a = \cos^2 R \) and \( R \) is the angle opposite the side \( r \).

1. Using the Law of Cosines, we have:

\[
\cos R = \frac{p^2 + q^2 - r^2}{2pq}
\]

2. Squaring both sides to find \( \cos^2 R \):

\[
a = \cos^2 R = \left( \frac{p^2 + q^2 - r^2}{2pq} \right)^2
\]

3. Expand the squared term:

\[
a = \cos^2 R = \frac{(p^2 + q^2 - r^2)^2}{(2pq)^2}
\]

4. Simplify the denominator:

\[
= \frac{(p^2 + q^2 - r^2)^2}{4p^2q^2}
\]

5. Expand the numerator:

\[
= \frac{p^4 + q^4 + r^4 + 2p^2q^2 - 2p^2r^2 - 2q^2r^2}{4p^2q^2}
\]

6. Substitute the given condition \( p^4 + q^4 + r^4 = 2r^2(p^2 + q

In [8]:
 # Train
try:
    trainer.train()
    logger.info("Training completed successfully")
except Exception as e:
    logger.error(f"Training failed: {str(e)}")
    wandb.finish()
    raise

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 7 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (7 x 4 x 1) = 28
 "-____-"     Trainable parameters = 262,144,000/14,921,651,200 (1.76% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


WARNING 03-09 21:06:01 scheduler.py:1754] Sequence group 6 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Available kwargs: ['prompts', 'id', 'problem', 'solution', 'source', 'answer', 'numeric_value', 'partial_solution', 'full_solution', 'is_correct', 'wrong_step', 'example_type']
example_type found: ['solution', 'solution', 'solution', 'solution', 'solution', 'solution', 'solution'] (type: <class 'list'>)
example_type list length: 7
First element: solution (type: <class 'str'>)
Extracted example types: {'solution': 7}
Type counts in batch: finalization=0, solution=7, programming=0, tutor=0
Selected solution reward (majority type or default)
Using solution reward for entire batch of 7 examples
Extracted example types: {'solution': 7}
Processing example type: solution with group_reward
Processing completion 1/7 in group
Used group_reward with result: 0.0000
Processing example type: solution with group_reward
Processing completion 2/7 in group
Used group_reward with result: 0.0000
Processing example type: solution with group_reward
Processing completion 3/7 in group
Used group_reward with r

Task was destroyed but it is pending!
task: <Task cancelling name='Task-10' coro=<Event.wait() running at /Home/stat/laschos/.conda/envs/sloth/lib/python3.11/asyncio/locks.py:213> wait_for=<Future cancelled>>


OutOfMemoryError: CUDA out of memory. Tried to allocate 188.00 MiB. GPU 0 has a total capacity of 39.39 GiB of which 20.88 MiB is free. Including non-PyTorch memory, this process has 39.34 GiB memory in use. Of the allocated memory 37.39 GiB is allocated by PyTorch, with 44.00 MiB allocated in private pools (e.g., CUDA Graphs), and 26.74 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Save Model

Finally, let's save the trained model.

In [ ]:
# Save model
try:
    models_dir = "models"
    os.makedirs(os.path.join(models_dir, reward_config.model_type), exist_ok=True)
    model_output_dir = os.path.join(models_dir, reward_config.model_type, timestamp)
    model.save_pretrained_merged(model_output_dir, tokenizer, save_method="merged_16bit")
    logger.info(f"Merged model saved to {model_output_dir}")
    print(f"Model saved to {model_output_dir}")
except Exception as e:
    logger.error(f"Failed to save model: {str(e)}")
    print(f"Error saving model: {str(e)}")
finally:
    wandb.finish()
    print("Wandb logging finished")

Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 476.04 out of 1007.58 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


  0%|                                           | 0/40 [00:00<?, ?it/s]
We will save to Disk and not RAM now.
100%|██████████████████████████████████| 40/40 [01:35<00:00,  2.39s/it]


Unsloth: Saving tokenizer... Done.
Done.


Merged model saved to models/dynamic_1/20250308_205221


Model saved to models/dynamic_1/20250308_205221


NameError: name 'use_wandb' is not defined

## Conclusion

This notebook has demonstrated the complete training process for a Qwen model using GRPO with dynamic rewards. We've seen how to:

1. Configure the reward function
2. Prepare a dataset with different example types
3. Initialize and configure the model with LoRA
4. Set up the GRPO training process
5. Train the model and visualize the results
6. Save the trained model

This interactive approach allows for better monitoring and understanding of the training process compared to running the script directly.